In [1]:
%%writefile dataset.py
from __future__ import annotations

import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset
from torchvision.transforms import InterpolationMode

N_SLOT = 6
CACHE_IMG = 336

TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

SILVER_LABEL_ACCURACY = {
    "ACL": 0.948,
    "MCL": 0.966,
    "Medial Meniscus": 0.879,
    "Lateral Meniscus": 0.879,
    "Medial OA": 0.897,
    "Lateral OA": 0.879,
    "PF OA": 0.931,
    "Effusion": 0.759,
    "Synovitis": 0.690,
    "Baker's": 0.931,
    "Contusion": 0.810,
    "Fracture": 0.793,
}

GOLD_WEIGHT_MULTIPLIER = 1.0


def infer_cache_shape(preprocessed_dir, split):
    preprocessed_dir = Path(preprocessed_dir)
    order_path = preprocessed_dir / f"{split}_study_order.csv"
    cache_path = preprocessed_dir / f"{split}_cache.dat"
    if not order_path.is_file():
        raise FileNotFoundError(f"{order_path} not found")
    if not cache_path.is_file():
        raise FileNotFoundError(f"{cache_path} not found")

    n_studies = len(pd.read_csv(order_path))
    cache_bytes = cache_path.stat().st_size
    per_study = N_SLOT * CACHE_IMG * CACHE_IMG
    if n_studies == 0 or cache_bytes % (n_studies * per_study) != 0:
        raise ValueError(
            f"{split}: {cache_path.name} is {cache_bytes} bytes, which doesn't divide "
            f"evenly by {n_studies} studies x {per_study} bytes/study - the cache file "
            f"looks truncated or corrupted")
    cache_slices = cache_bytes // (n_studies * per_study)
    return (n_studies, N_SLOT, cache_slices, CACHE_IMG, CACHE_IMG)


class KneeMRICache:

    def __init__(self, preprocessed_dir, split):
        preprocessed_dir = Path(preprocessed_dir)
        shape = infer_cache_shape(preprocessed_dir, split)
        self.cache = np.memmap(
            preprocessed_dir / f"{split}_cache.dat",
            dtype=np.uint8,
            mode="r",
            shape=shape,
        )
        self.mask = np.load(preprocessed_dir / f"{split}_mask.npy")
        self.study_order = pd.read_csv(preprocessed_dir / f"{split}_study_order.csv")[
            "StudyInstanceUID"
        ].tolist()
        self.study_to_idx = {s: i for i, s in enumerate(self.study_order)}
        if self.mask.shape[0] != len(self.study_order):
            raise ValueError(
                f"{split}: mask has {self.mask.shape[0]} rows but "
                f"study_order has {len(self.study_order)} entries"
            )


def augment_slots(slots):
    S = slots.shape[0]
    for i in range(S):
        img = slots[i]
        if random.random() < 0.5:
            angle = random.uniform(-12, 12)
            img = TF.rotate(img, angle, interpolation=InterpolationMode.BILINEAR)
        if random.random() < 0.5:
            img = TF.adjust_brightness(img, random.uniform(0.85, 1.15))
        if random.random() < 0.5:
            img = TF.adjust_contrast(img, random.uniform(0.85, 1.15))
        slots[i] = img
    return slots


class KneeMRIDataset(Dataset):
    def __init__(self, cache: KneeMRICache, study_uids, labels_df=None, augment=False):
        self.cache = cache
        self.study_uids = list(study_uids)
        self.labels_df = None
        self.weights = None
        self.augment = augment
        if labels_df is not None:
            self.labels_df = labels_df.set_index("StudyInstanceUID")
            reg_acc = np.array(
                [SILVER_LABEL_ACCURACY[t] for t in TARGETS], dtype=np.float32
            )
            self._silver_weight = reg_acc
            self._gold_weight = np.full(
                len(TARGETS), GOLD_WEIGHT_MULTIPLIER, dtype=np.float32
            )

    def __len__(self):
        return len(self.study_uids)

    def __getitem__(self, i):
        uid = self.study_uids[i]
        idx = self.cache.study_to_idx[uid]
        slots = torch.from_numpy(np.asarray(self.cache.cache[idx])).clone()
        mask = torch.from_numpy(self.cache.mask[idx].copy())

        if self.augment:
            slots = augment_slots(slots)

        if self.labels_df is None:
            return uid, slots, mask

        row = self.labels_df.loc[uid]
        labels = torch.tensor([float(row[t]) for t in TARGETS], dtype=torch.float32)
        is_gold = str(row.get("label_source", "")).lower() == "gold"
        weight = self._gold_weight if is_gold else self._silver_weight
        weight = torch.from_numpy(weight.copy())
        return uid, slots, mask, labels, weight


def build_splits(
    preprocessed_dir, labels_csv, train_csv, val_silver_frac=0.1,
    seed=2026, gold_folds=5, gold_fold=0,
):
    if not (0 <= gold_fold < gold_folds):
        raise ValueError(f"gold_fold={gold_fold} must be in [0, {gold_folds})")

    labels_df = pd.read_csv(labels_csv)
    cache_study_set = set(
        pd.read_csv(Path(preprocessed_dir) / "train_study_order.csv")[
            "StudyInstanceUID"
        ]
    )
    labels_df = labels_df[labels_df["StudyInstanceUID"].isin(cache_study_set)].copy()

    reports = pd.read_csv(train_csv, usecols=["StudyInstanceUID", "Report"])
    labels_df = labels_df.merge(reports, on="StudyInstanceUID", how="left")
    labels_df["Report"] = labels_df["Report"].fillna("")

    group_of = dict(zip(labels_df["StudyInstanceUID"], labels_df["Report"]))
    source_of = dict(zip(labels_df["StudyInstanceUID"], labels_df["label_source"]))

    groups = {}
    for uid, rep in group_of.items():
        groups.setdefault(rep, []).append(uid)

    gold_groups, silver_only_groups = [], []
    for rep, uids in groups.items():
        if any(source_of[u] == "gold" for u in uids):
            gold_groups.append(uids)
        else:
            silver_only_groups.append(uids)

    rng = np.random.RandomState(seed)
    rng.shuffle(gold_groups)
    rng.shuffle(silver_only_groups)

    val_gold_uids = []
    for i, uids in enumerate(gold_groups):
        if i % gold_folds == gold_fold:
            val_gold_uids.extend(uids)

    n_silver_total = sum(1 for u in group_of if source_of[u] != "gold")
    silver_target = int(round(n_silver_total * val_silver_frac))

    val_silver_uids = []
    n_silver_in_val = 0
    for uids in silver_only_groups:
        if n_silver_in_val >= silver_target:
            break
        val_silver_uids.extend(uids)
        n_silver_in_val += len(uids)

    val_uids = set(val_gold_uids) | set(val_silver_uids)
    all_uids = set(group_of.keys())
    train_uids = list(all_uids - val_uids)
    rng.shuffle(train_uids)

    return train_uids, val_gold_uids, val_silver_uids, labels_df

Writing dataset.py


In [2]:
%%writefile model.py
from __future__ import annotations

from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

DINOV2_CONFIGS = {
    "small": dict(
        hidden_size=384, num_hidden_layers=12, num_attention_heads=6, mlp_ratio=4,
        image_size=518, patch_size=14,
    ),
    "base": dict(
        hidden_size=768, num_hidden_layers=12, num_attention_heads=12, mlp_ratio=4,
        image_size=518, patch_size=14,
    ),
    "large": dict(
        hidden_size=1024, num_hidden_layers=24, num_attention_heads=16, mlp_ratio=4,
        image_size=518, patch_size=14,
    ),
}

DINOV2_HUB_IDS = {
    "small": "facebook/dinov2-small",
    "base": "facebook/dinov2-base",
    "large": "facebook/dinov2-large",
}


class ResNet50SlotEncoder(nn.Module):
    def __init__(self, out_dim=512, pretrained=True):
        super().__init__()
        weights = (
            torchvision.models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        )
        backbone = torchvision.models.resnet50(weights=weights)
        self.body = nn.Sequential(*list(backbone.children())[:-1])
        self.proj = nn.Linear(2048, out_dim)

    def forward(self, x):
        feat = self.body(x).flatten(1)
        return self.proj(feat)


def find_dinov2_checkpoint(variant="small", search_root="/kaggle/input"):
    root = Path(search_root)
    if not root.is_dir():
        return None
    hits = []
    for path in root.rglob("config.json"):
        if "dinov2" in str(path.parent).lower():
            hits.append(path.parent)
    for h in hits:
        if variant in str(h).lower():
            return h
    return hits[0] if hits else None


class Dinov2SlotEncoder(nn.Module):
    def __init__(
        self,
        out_dim=512,
        pretrained=True,
        variant="small",
        unfreeze_last=2,
        source=None,
    ):
        super().__init__()
        from transformers import AutoModel, Dinov2Config, Dinov2Model

        if pretrained:
            path = source or find_dinov2_checkpoint(variant) or DINOV2_HUB_IDS[variant]
            self.backbone = AutoModel.from_pretrained(str(path))
        else:
            cfg = Dinov2Config(**DINOV2_CONFIGS[variant])
            self.backbone = Dinov2Model(cfg)

        n_layers = len(self.backbone.encoder.layer)
        for p in self.backbone.parameters():
            p.requires_grad = False
        for blk in self.backbone.encoder.layer[max(0, n_layers - unfreeze_last) :]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in self.backbone.layernorm.parameters():
            p.requires_grad = True

        hidden_size = self.backbone.config.hidden_size
        self.proj = nn.Linear(hidden_size, out_dim)

    def forward(self, x):
        out = self.backbone(pixel_values=x, interpolate_pos_encoding=True)
        cls_token = out.last_hidden_state[:, 0]
        return self.proj(cls_token)


def build_encoder(backbone, out_dim=512, pretrained=True, **kwargs):
    if backbone == "resnet50":
        return ResNet50SlotEncoder(out_dim=out_dim, pretrained=pretrained)
    if backbone == "dinov2":
        return Dinov2SlotEncoder(out_dim=out_dim, pretrained=pretrained, **kwargs)
    raise ValueError(f"unknown backbone {backbone!r}, expected 'resnet50' or 'dinov2'")


class KneeMRIModel(nn.Module):
    def __init__(
        self,
        n_labels=12,
        feat_dim=512,
        backbone="resnet50",
        pretrained=True,
        train_img_size=224,
        backbone_kwargs=None,
    ):
        super().__init__()
        self.encoder = build_encoder(
            backbone, out_dim=feat_dim, pretrained=pretrained, **(backbone_kwargs or {})
        )
        self.backbone_name = backbone
        self.train_img_size = train_img_size
        self.classifier = nn.Sequential(
            nn.LayerNorm(feat_dim * 2),
            nn.Linear(feat_dim * 2, feat_dim),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(feat_dim, n_labels),
        )
        self.register_buffer(
            "mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        )
        self.register_buffer(
            "std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        )

    def forward(self, slots, mask):
        B, S, C, H, W = slots.shape
        x = slots.reshape(B * S, C, H, W).float() / 255.0
        if x.shape[-1] != self.train_img_size:
            x = F.interpolate(
                x,
                size=(self.train_img_size, self.train_img_size),
                mode="bilinear",
                align_corners=False,
            )
        x = (x - self.mean) / self.std

        feat = self.encoder(x).reshape(B, S, -1)

        mask_f = mask.unsqueeze(-1)
        mean_feat = (feat * mask_f).sum(1) / mask_f.sum(1).clamp(min=1.0)

        neg_inf = torch.finfo(feat.dtype).min
        masked_for_max = feat.masked_fill(mask_f == 0, neg_inf)
        max_feat = masked_for_max.max(1).values
        no_slots = (mask.sum(1) == 0).unsqueeze(-1)
        max_feat = torch.where(no_slots, torch.zeros_like(max_feat), max_feat)

        pooled = torch.cat([mean_feat, max_feat], dim=-1)
        return self.classifier(pooled)

Writing model.py


In [3]:
%%writefile train.py
from __future__ import annotations

import argparse
import glob
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score

sys.path.insert(0, str(Path(__file__).parent))
from dataset import KneeMRICache, KneeMRIDataset, build_splits, TARGETS
from model import KneeMRIModel

DEFAULT_PREPROCESSED_DIR = "preprocessed"
DEFAULT_LABELS_CSV = "report_labels_v1_regex.csv"
OUT_DIR = Path("training_output")
TIME_BUDGET_S = 8.5 * 3600
BATCH_SIZE = 16
NUM_WORKERS = 4
BACKBONE_LR = 1e-5
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 40
PATIENCE = 10
WARMUP_EPOCHS = 2
GRAD_CLIP = 5.0


def warn_if_interactive():
    run_type = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "")
    if run_type and run_type.lower() != "batch":
        print(
            f"[WARNING] KAGGLE_KERNEL_RUN_TYPE={run_type!r} - this looks like an "
            f"interactive Draft Session. /kaggle/working here is EPHEMERAL and will "
            f"not persist. Use 'Save Version -> Save & Run All (Commit)' to get a "
            f"run whose output actually survives.",
            flush=True,
        )


def collate_labeled(batch):
    uids, slots, mask, labels, weight = zip(*batch)
    return (
        list(uids),
        torch.stack(slots),
        torch.stack(mask),
        torch.stack(labels),
        torch.stack(weight),
    )


def make_loader(cache, uids, labels_df, batch_size, shuffle, augment=False):
    ds = KneeMRIDataset(cache, uids, labels_df, augment=augment)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        collate_fn=collate_labeled,
        drop_last=False,
    )


def compute_pos_weight(labels_df, uids):
    sub = labels_df.set_index("StudyInstanceUID").loc[uids]
    pos = sub[TARGETS].sum(axis=0).values.astype(np.float32)
    neg = len(uids) - pos
    pos = np.clip(pos, 1, None)
    return torch.from_numpy(neg / pos)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_logits, all_labels = [], []
    for uids, slots, mask, labels, _ in loader:
        slots, mask = slots.to(device), mask.to(device)
        logits = model(slots, mask)
        all_logits.append(logits.cpu().numpy())
        all_labels.append(labels.numpy())
    if not all_logits:
        return float("nan"), {}
    logits = np.concatenate(all_logits, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    per_label_auc = {}
    for i, t in enumerate(TARGETS):
        y_hard = (labels[:, i] >= 0.5).astype(int)
        if len(np.unique(y_hard)) < 2:
            continue
        try:
            per_label_auc[t] = roc_auc_score(y_hard, logits[:, i])
        except ValueError:
            continue
    mean_auc = (
        float(np.mean(list(per_label_auc.values()))) if per_label_auc else float("nan")
    )
    return mean_auc, per_label_auc


def save_checkpoint(path, model, optimizer, scaler, epoch, best_auc, rng_state):
    tmp = Path(str(path) + ".tmp")
    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scaler": scaler.state_dict(),
            "epoch": epoch,
            "best_auc": best_auc,
            "torch_rng_state": rng_state,
        },
        tmp,
    )
    os.replace(tmp, path)


def load_checkpoint(path, model, optimizer, scaler, device):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scaler.load_state_dict(ckpt["scaler"])
    torch.set_rng_state(ckpt["torch_rng_state"])
    return ckpt["epoch"], ckpt["best_auc"]


def aggregate_oof(pattern):
    paths = sorted(glob.glob(pattern))
    if not paths:
        raise FileNotFoundError(f"no files matched {pattern!r}")
    df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
    dup = df["StudyInstanceUID"].duplicated()
    if dup.any():
        raise ValueError(
            f"{dup.sum()} StudyInstanceUID(s) appear in more than one fold's "
            f"held-out set - folds should partition the gold studies exactly "
            f"once each. Check --gold-folds/--gold-fold matched across all runs "
            f"that produced files matching {pattern!r}."
        )
    print(
        f"[INFO] pooled {len(df)} gold studies from {len(paths)} fold file(s)",
        flush=True,
    )
    per_label_auc = {}
    for t in TARGETS:
        y = df[f"true_{t}"].values
        p = df[f"pred_{t}"].values
        if len(np.unique(y)) < 2:
            continue
        per_label_auc[t] = roc_auc_score(y, p)
    mean_auc = (
        float(np.mean(list(per_label_auc.values()))) if per_label_auc else float("nan")
    )
    print(f"[AGGREGATE OOF GOLD] mean_auc={mean_auc:.4f}", flush=True)
    for t, a in per_label_auc.items():
        print(f"    {t}: {a:.4f}", flush=True)
    return mean_auc, per_label_auc


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--preprocessed-dir", default=DEFAULT_PREPROCESSED_DIR)
    ap.add_argument("--labels-csv", default=DEFAULT_LABELS_CSV)
    ap.add_argument("--train-csv", required=True)
    ap.add_argument("--out-dir", default=str(OUT_DIR))
    ap.add_argument("--batch-size", type=int, default=BATCH_SIZE)
    ap.add_argument("--max-epochs", type=int, default=MAX_EPOCHS)
    ap.add_argument("--time-budget-s", type=float, default=TIME_BUDGET_S)
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--pretrained", action="store_true", default=True)
    ap.add_argument("--no-pretrained", dest="pretrained", action="store_false")
    ap.add_argument("--img-size", type=int, default=224)
    ap.add_argument("--backbone", choices=["resnet50", "dinov2"], default="resnet50")
    ap.add_argument(
        "--dinov2-variant", choices=["small", "base", "large"], default="small"
    )
    ap.add_argument("--dinov2-source", default=None)
    ap.add_argument("--dinov2-unfreeze-last", type=int, default=2)
    ap.add_argument("--gold-folds", type=int, default=5)
    ap.add_argument("--gold-fold", type=int, default=0)
    ap.add_argument("--aggregate-oof", default=None)
    args = ap.parse_args()

    if args.aggregate_oof:
        aggregate_oof(args.aggregate_oof)
        return

    warn_if_interactive()
    start_time = time.time()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    log_path = out_dir / "train_log.jsonl"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[INFO] device={device}", flush=True)

    train_uids, val_gold_uids, val_silver_uids, labels_df = build_splits(
        args.preprocessed_dir, args.labels_csv, args.train_csv,
        gold_folds=args.gold_folds, gold_fold=args.gold_fold,
    )
    print(
        f"[INFO] gold_fold={args.gold_fold}/{args.gold_folds} "
        f"train={len(train_uids)} val_gold={len(val_gold_uids)} "
        f"val_silver={len(val_silver_uids)}",
        flush=True,
    )
    if len(val_gold_uids) == 0:
        print(
            "[WARNING] no gold studies available for validation - checkpoint "
            "selection will fall back to silver AUC, which is a noisier signal "
            "since silver labels are regex-derived, not radiologist-derived.",
            flush=True,
        )

    cache = KneeMRICache(args.preprocessed_dir, "train")

    train_loader = make_loader(
        cache, train_uids, labels_df, args.batch_size, shuffle=True, augment=True
    )
    val_gold_loader = (
        make_loader(cache, val_gold_uids, labels_df, args.batch_size, shuffle=False)
        if val_gold_uids
        else None
    )
    val_silver_loader = (
        make_loader(cache, val_silver_uids, labels_df, args.batch_size, shuffle=False)
        if val_silver_uids
        else None
    )

    pos_weight = compute_pos_weight(labels_df, train_uids).to(device)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction="none")

    backbone_kwargs = {}
    if args.backbone == "dinov2":
        backbone_kwargs = {
            "variant": args.dinov2_variant,
            "unfreeze_last": args.dinov2_unfreeze_last,
            "source": args.dinov2_source,
        }
    model = KneeMRIModel(
        n_labels=len(TARGETS),
        backbone=args.backbone,
        pretrained=args.pretrained,
        train_img_size=args.img_size,
        backbone_kwargs=backbone_kwargs,
    ).to(device)

    pretrained_submodule = (
        model.encoder.body if args.backbone == "resnet50" else model.encoder.backbone
    )
    backbone_params = [p for p in pretrained_submodule.parameters() if p.requires_grad]
    head_params = list(model.encoder.proj.parameters()) + list(
        model.classifier.parameters()
    )
    n_backbone_trainable = sum(p.numel() for p in backbone_params)
    n_head_trainable = sum(p.numel() for p in head_params)
    print(
        f"[INFO] backbone={args.backbone} backbone_trainable_params="
        f"{n_backbone_trainable/1e6:.1f}M head_trainable_params={n_head_trainable/1e6:.1f}M",
        flush=True,
    )
    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": BACKBONE_LR},
            {"params": head_params, "lr": HEAD_LR},
        ],
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, args.max_epochs - WARMUP_EPOCHS)
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    start_epoch = 0
    best_auc = -1.0
    ckpt_path = out_dir / "last.pt"
    best_path = out_dir / "best.pt"
    if args.resume and ckpt_path.is_file():
        start_epoch, best_auc = load_checkpoint(
            ckpt_path, model, optimizer, scaler, device
        )
        start_epoch += 1
        print(
            f"[INFO] resumed from epoch {start_epoch}, best_auc so far={best_auc:.4f}",
            flush=True,
        )

    epochs_since_improve = 0
    stopped_reason = None

    for epoch in range(start_epoch, args.max_epochs):
        if time.time() - start_time > args.time_budget_s:
            stopped_reason = "time_budget"
            print(
                f"[INFO] time budget ({args.time_budget_s}s) reached before epoch "
                f"{epoch}, stopping gracefully",
                flush=True,
            )
            break

        if epoch < WARMUP_EPOCHS:
            warmup_scale = (epoch + 1) / WARMUP_EPOCHS
            for g, base_lr in zip(optimizer.param_groups, [BACKBONE_LR, HEAD_LR]):
                g["lr"] = base_lr * warmup_scale

        model.train()
        running_loss, n_batches = 0.0, 0
        for uids, slots, mask, labels, weight in train_loader:
            slots, mask = slots.to(device), mask.to(device)
            labels, weight = labels.to(device), weight.to(device)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type=device.type, enabled=(device.type == "cuda")
            ):
                logits = model(slots, mask)
                per_elem = bce(logits, labels)
                loss = (per_elem * weight).sum() / weight.sum().clamp(min=1.0)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            n_batches += 1

        if epoch >= WARMUP_EPOCHS:
            scheduler.step()

        train_loss = running_loss / max(1, n_batches)
        gold_auc, gold_per_label = (
            (float("nan"), {})
            if val_gold_loader is None
            else evaluate(model, val_gold_loader, device)
        )
        silver_auc, _ = (
            (float("nan"), {})
            if val_silver_loader is None
            else evaluate(model, val_silver_loader, device)
        )

        if val_gold_loader is not None and val_silver_loader is not None:
            selection_auc = 0.5 * gold_auc + 0.5 * silver_auc
        elif val_gold_loader is not None:
            selection_auc = gold_auc
        else:
            selection_auc = silver_auc

        elapsed = time.time() - start_time
        print(
            f"[epoch {epoch}] loss={train_loss:.4f} gold_auc={gold_auc:.4f} "
            f"silver_auc={silver_auc:.4f} selection_auc={selection_auc:.4f} "
            f"elapsed={elapsed/60:.1f}min",
            flush=True,
        )

        with open(log_path, "a") as f:
            f.write(
                json.dumps(
                    {
                        "epoch": epoch,
                        "train_loss": train_loss,
                        "gold_auc": gold_auc,
                        "silver_auc": silver_auc,
                        "selection_auc": selection_auc,
                        "gold_per_label_auc": gold_per_label,
                        "elapsed_s": elapsed,
                    }
                )
                + "\n"
            )

        rng_state = torch.get_rng_state()
        save_checkpoint(ckpt_path, model, optimizer, scaler, epoch, best_auc, rng_state)

        if not np.isnan(selection_auc) and selection_auc > best_auc:
            best_auc = selection_auc
            epochs_since_improve = 0
            save_checkpoint(
                best_path, model, optimizer, scaler, epoch, best_auc, rng_state
            )
            print(
                f"[epoch {epoch}] new best (auc={best_auc:.4f}) -> saved {best_path}",
                flush=True,
            )
        else:
            epochs_since_improve += 1

        if epochs_since_improve >= PATIENCE:
            stopped_reason = "early_stop"
            print(f"[INFO] no improvement for {PATIENCE} epochs, stopping", flush=True)
            break
    else:
        stopped_reason = "max_epochs"

    if not best_path.is_file():
        raise RuntimeError(
            "training finished without ever saving best.pt - this means validation "
            "AUC was NaN every epoch (no usable val labels), which means the run "
            "produced no usable model. Check that val_gold_uids / labels_df are "
            "non-empty before trusting any downstream inference."
        )

    if val_gold_loader is not None:
        best_ckpt = torch.load(best_path, map_location=device)
        model.load_state_dict(best_ckpt["model"])
        model.eval()
        oof_rows = []
        with torch.no_grad():
            for uids, slots, mask, labels, _ in val_gold_loader:
                slots, mask = slots.to(device), mask.to(device)
                probs = torch.sigmoid(model(slots, mask)).cpu().numpy()
                labels_np = labels.numpy()
                for uid, p, y in zip(uids, probs, labels_np):
                    row = {"StudyInstanceUID": uid, "gold_fold": args.gold_fold}
                    for i, t in enumerate(TARGETS):
                        row[f"pred_{t}"] = float(p[i])
                        row[f"true_{t}"] = float(y[i])
                    oof_rows.append(row)
        oof_path = out_dir / f"oof_gold_fold{args.gold_fold}.csv"
        pd.DataFrame(oof_rows).to_csv(oof_path, index=False)
        print(
            f"[INFO] wrote {len(oof_rows)} held-out gold predictions -> {oof_path} "
            f"(pool across folds with --aggregate-oof once every fold is done)",
            flush=True,
        )

    summary = {
        "stopped_reason": stopped_reason,
        "best_auc": best_auc,
        "final_epoch": epoch,
        "elapsed_s": time.time() - start_time,
        "gold_fold": args.gold_fold,
        "gold_folds": args.gold_folds,
    }
    (out_dir / "_TRAIN_MANIFEST.json").write_text(json.dumps(summary, indent=2))
    print(f"[DONE] {summary}", flush=True)


if __name__ == "__main__":
    main()

Writing train.py


In [4]:
!python train.py \
  --preprocessed-dir /kaggle/input/notebooks/varunhittuvalli/notebook539c2deb60/preprocessed \
  --labels-csv /kaggle/input/datasets/varunhittuvalli/report-labels-v2-csv/report_labels_v2.csv \
  --train-csv /kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv \
  --out-dir training_output_dinov2_fold0 \
  --backbone dinov2 \
  --dinov2-variant small \
  --gold-fold 0 \
  --gold-folds 5

[INFO] device=cuda
[INFO] gold_fold=0/5 train=3958 val_gold=13 val_silver=435
Loading weights: 100%|█| 223/223 [00:00<00:00, 3045.32it/s, Materializing param=
[INFO] backbone=dinov2 backbone_trainable_params=3.6M head_trainable_params=0.7M
/kaggle/working/train.py:282: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/kaggle/working/dataset.py:131: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  slots = torch.from_numpy(np.asarray(self.cache.cache[idx])).clone()
/kaggle/wo

In [5]:
!python train.py \
  --preprocessed-dir /kaggle/input/notebooks/varunhittuvalli/notebook539c2deb60/preprocessed \
  --labels-csv /kaggle/input/datasets/varunhittuvalli/report-labels-v2-csv/report_labels_v2.csv \
  --train-csv /kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv \
  --out-dir training_output_dinov2_fold1 \
  --backbone dinov2 \
  --dinov2-variant small \
  --gold-fold 1 \
  --gold-folds 5

[INFO] device=cuda
[INFO] gold_fold=1/5 train=3959 val_gold=12 val_silver=435
Loading weights: 100%|█| 223/223 [00:00<00:00, 2573.72it/s, Materializing param=
[INFO] backbone=dinov2 backbone_trainable_params=3.6M head_trainable_params=0.7M
/kaggle/working/train.py:282: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/kaggle/working/dataset.py:131: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  slots = torch.from_numpy(np.asarray(self.cache.cache[idx])).clone()
/kaggle/wo

In [6]:
!python train.py \
  --preprocessed-dir /kaggle/input/notebooks/varunhittuvalli/notebook539c2deb60/preprocessed \
  --labels-csv /kaggle/input/datasets/varunhittuvalli/report-labels-v2-csv/report_labels_v2.csv \
  --train-csv /kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv \
  --out-dir training_output_dinov2_fold2 \
  --backbone dinov2 \
  --dinov2-variant small \
  --gold-fold 2 \
  --gold-folds 5

[INFO] device=cuda
[INFO] gold_fold=2/5 train=3959 val_gold=12 val_silver=435
Loading weights: 100%|█| 223/223 [00:00<00:00, 2830.64it/s, Materializing param=
[INFO] backbone=dinov2 backbone_trainable_params=3.6M head_trainable_params=0.7M
/kaggle/working/train.py:282: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/kaggle/working/dataset.py:131: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  slots = torch.from_numpy(np.asarray(self.cache.cache[idx])).clone()
/kaggle/wo

In [7]:
!python train.py \
  --preprocessed-dir /kaggle/input/notebooks/varunhittuvalli/notebook539c2deb60/preprocessed \
  --labels-csv /kaggle/input/datasets/varunhittuvalli/report-labels-v2-csv/report_labels_v2.csv \
  --train-csv /kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv \
  --out-dir training_output_dinov2_fold3 \
  --backbone dinov2 \
  --dinov2-variant small \
  --gold-fold 3 \
  --gold-folds 5

[INFO] device=cuda
[INFO] gold_fold=3/5 train=3960 val_gold=11 val_silver=435
Loading weights: 100%|█| 223/223 [00:00<00:00, 2742.84it/s, Materializing param=
[INFO] backbone=dinov2 backbone_trainable_params=3.6M head_trainable_params=0.7M
/kaggle/working/train.py:282: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/kaggle/working/dataset.py:131: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  slots = torch.from_numpy(np.asarray(self.cache.cache[idx])).clone()
/kaggle/wo

In [8]:
!python train.py \
  --preprocessed-dir /kaggle/input/notebooks/varunhittuvalli/notebook539c2deb60/preprocessed \
  --labels-csv /kaggle/input/datasets/varunhittuvalli/report-labels-v2-csv/report_labels_v2.csv \
  --train-csv /kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv \
  --out-dir training_output_dinov2_fold4 \
  --backbone dinov2 \
  --dinov2-variant small \
  --gold-fold 4 \
  --gold-folds 5

[INFO] device=cuda
[INFO] gold_fold=4/5 train=3960 val_gold=11 val_silver=435
Loading weights: 100%|█| 223/223 [00:00<00:00, 2451.03it/s, Materializing param=
[INFO] backbone=dinov2 backbone_trainable_params=3.6M head_trainable_params=0.7M
/kaggle/working/train.py:282: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/kaggle/working/dataset.py:131: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  slots = torch.from_numpy(np.asarray(self.cache.cache[idx])).clone()
/kaggle/wo

In [9]:
!python train.py --aggregate-oof "training_output_dinov2_fold*/oof_gold_fold*.csv"

usage: train.py [-h] [--preprocessed-dir PREPROCESSED_DIR]
                [--labels-csv LABELS_CSV] --train-csv TRAIN_CSV
                [--out-dir OUT_DIR] [--batch-size BATCH_SIZE]
                [--max-epochs MAX_EPOCHS] [--time-budget-s TIME_BUDGET_S]
                [--resume] [--pretrained] [--no-pretrained]
                [--img-size IMG_SIZE] [--backbone {resnet50,dinov2}]
                [--dinov2-variant {small,base,large}]
                [--dinov2-source DINOV2_SOURCE]
                [--dinov2-unfreeze-last DINOV2_UNFREEZE_LAST]
                [--gold-folds GOLD_FOLDS] [--gold-fold GOLD_FOLD]
                [--aggregate-oof AGGREGATE_OOF]
train.py: error: the following arguments are required: --train-csv


In [10]:
%%writefile image_preprocessing.py
from __future__ import annotations
import argparse
import gc
import json
import os
import re
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn.functional as F
T0 = time.time()
def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)
CROP_MM = 130.0
CACHE_IMG = 336
CACHE_SLICES = 3
SLICE_BAND = (0.20, 0.80)
LAT_MIN_OFFSET_MM = 20.0
HDR_THREADS = 16
PIX_THREADS = 12
ORDER_THREADS = 32
ORDER_BUDGET_S = 5400
FLUSH_EVERY = 500
SLOTS = [
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
]
N_SLOT = len(SLOTS)
FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}
_SEP = re.compile(r"[_\-.]")
_FATSAT_RX = re.compile(r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|"
                        r"water excit|\btirm\b|\bsting\b|\bfatsup\b")
_T1_RX = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")
HDR_TAGS = ["SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
            "RepetitionTime", "EchoTime", "Laterality", "PixelSpacing", "Rows",
            "Columns", "RescaleSlope", "RescaleIntercept",
            "ImagePositionPatient", "ImageOrientationPatient"]
ORDER_TAGS = [(0x0020, 0x0032), (0x0020, 0x0037), (0x0020, 0x0013)]
DECODE_FAILED = []
def find_root():
    for c in [Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
              Path("/kaggle/input/rsna-knee-abnormality-detection"),
              Path("data"), Path(".")]:
        if (c / "test.csv").is_file() and (c / "test_series").is_dir():
            return c
    base = Path("/kaggle/input")
    if base.is_dir():
        for depth1 in sorted(p for p in base.iterdir() if p.is_dir()):
            for cand in [depth1] + sorted(p for p in depth1.iterdir() if p.is_dir()):
                if (cand / "test.csv").is_file():
                    return cand
    raise FileNotFoundError(
        f"competition mount not found (cwd {Path.cwd()}); expected a directory holding "
        f"test.csv and test_series/")
def available_gb():
    try:
        with open("/proc/meminfo") as fh:
            info = {k.strip(): v for k, v in
                    (l.split(":", 1) for l in fh if ":" in l)}
        return int(info["MemAvailable"].split()[0]) / 1024 ** 2
    except Exception:
        return 16.0
def plan_cache(n_study, cache_fraction=0.45, budget_max_gb=24.0):
    avail = available_gb()
    budget = min(avail * cache_fraction, budget_max_gb)
    per_slice = n_study * N_SLOT * CACHE_IMG * CACHE_IMG
    afford = int(budget * 1024 ** 3 // max(per_slice, 1))
    slices = max(1, min(CACHE_SLICES, afford))
    log(f"memory: {avail:.1f} GB available, {budget:.1f} GB budgeted for the cache "
        f"-> {slices} slices/slot" + (f" (wanted {CACHE_SLICES})" if slices < CACHE_SLICES else ""))
    return slices
def _hdr_vec(s, n):
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split("|")]
    except ValueError:
        return None
    return np.array(v) if len(v) >= n else None
def probe(item):
    split, study, series, path = item
    row = {"split": split, "StudyInstanceUID": study, "SeriesInstanceUID": series, "dir": path}
    try:
        files = sorted(e.name for e in os.scandir(path) if e.name.endswith(".dcm"))
        row["files"] = files
        row["n_slices"] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]),
                             stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":
                row[t] = "|".join(str(x) for x in v)
            else:
                row[t] = str(v)
    except Exception as exc:
        row["err"] = str(exc)[:120]
    return row
def walk(root, split):
    base = root / f"{split}_series"
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=["split", "StudyInstanceUID", "SeriesInstanceUID",
                                     "dir", "files", "n_slices"] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)
def annotate(df):
    desc = (df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna(""))
    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)
    opts = df["ScanOptions"].fillna("").str.upper().str.split("|")
    opts_fs = opts.apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    df["fatsat"] = desc.str.contains(_FATSAT_RX) | opts_fs
    tr = pd.to_numeric(df["RepetitionTime"], errors="coerce")
    te = pd.to_numeric(df["EchoTime"], errors="coerce")
    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1, t2, pdw = desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX)
    df["weight"] = np.where(t1 & ~t2 & ~pdw, "T1",
                     np.where(t2 & ~pdw, "T2",
                       np.where(pdw, "PD",
                         np.where(gre, "GRE",
                           np.where(tr < 800, "T1",
                             np.where(te > 60, "T2",
                               np.where(tr >= 800, "PD", "UNK")))))))
    df["fluid"] = np.isin(df["weight"], ["PD", "T2"])
    df["px"] = pd.to_numeric(
        df["PixelSpacing"].fillna("").str.split("|").str[0].replace("", np.nan),
        errors="coerce")
    return df
def pick_slots(series_df, plane_map):
    series_df = series_df.copy()
    series_df["plane"] = series_df["SeriesInstanceUID"].map(plane_map)
    out = {}
    for study, g in series_df.groupby("StudyInstanceUID"):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g["plane"] == plane) & (g["fatsat"] == fs)
            if fluid is not None:
                sel &= (g["fluid"] == fluid)
            cand = g[sel]
            if len(cand):
                chosen[name] = cand.sort_values("n_slices", ascending=False).iloc[0]
        out[study] = chosen
    return out
def _natural_key(name):
    return tuple(int(x) if x.isdigit() else x.lower()
                 for x in re.split(r"(\d+)", str(name)))
def order_slices(rec):
    files, d = rec["files"], rec["dir"]
    keyed = []
    for f in files:
        k = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True,
                                 specific_tags=ORDER_TAGS)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            try:
                k = float(ds.InstanceNumber)
            except Exception:
                k = None
        keyed.append((k, f))
    if any(k is None for k, _ in keyed):
        return files, False
    return [f for _, f in sorted(keyed, key=lambda t: t[0])], True
def side_from_geometry(h):
    cx = {}
    for r in h.itertuples(index=False):
        ipp = _hdr_vec(getattr(r, "ImagePositionPatient", None), 3)
        iop = _hdr_vec(getattr(r, "ImageOrientationPatient", None), 6)
        ps = _hdr_vec(getattr(r, "PixelSpacing", None), 2)
        rows, cols = getattr(r, "Rows", None), getattr(r, "Columns", None)
        if ipp is None or iop is None or ps is None or not rows or not cols:
            continue
        try:
            c = ipp[:3] + iop[:3] * ps[1] * float(cols) / 2 + iop[3:6] * ps[0] * float(rows) / 2
        except (TypeError, ValueError):
            continue
        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))
    out = {}
    for st, xs in cx.items():
        m = float(np.median(xs))
        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else ("R" if m < 0 else "L")
    return out
def lat_of(h, tag=""):
    geo = side_from_geometry(h)
    d, n_tag, n_geo, n_none, n_disagree = {}, 0, 0, 0, 0
    for st, g in h.groupby("StudyInstanceUID"):
        v = [str(x).strip().upper() for x in g["Laterality"].dropna()]
        v = [x[0] for x in v if x and x[0] in ("L", "R")]
        side = v[0] if v else None
        if side is not None:
            n_tag += 1
            if geo.get(st) is not None and geo[st] != side:
                n_disagree += 1
        else:
            side = geo.get(st)
            n_geo += side is not None
            n_none += side is None
        d[st] = side
    log(f"{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, "
        f"{n_none} unresolved; tag and geometry disagree on {n_disagree} "
        f"({n_disagree / max(n_tag, 1):.1%} of the tagged)")
    return d
def normalise_laterality(img, plane, lat):
    if lat != "R":
        return img
    if plane in ("Coronal", "Axial"):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])
def read_slot(rec, n_slice, out_size):
    files, d, px = rec.get("ordered") or rec["files"], rec["dir"], rec["px"]
    n = len(files)
    if n == 0:
        return None
    lo, hi = int(SLICE_BAND[0] * (n - 1)), int(SLICE_BAND[1] * (n - 1))
    idx = np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo else np.array([n // 2])
    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])
    planes, errors = [], []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, "RescaleSlope", 1) or 1)
            ic = float(getattr(ds, "RescaleIntercept", 0) or 0)
            a = a * sl + ic
        except Exception as exc:
            a = None
            errors.append(f"{type(exc).__name__}: {str(exc)[:120]}")
        planes.append(a)
    got = [k for k, p in enumerate(planes) if p is not None]
    if not got:
        DECODE_FAILED.append({"series": rec.get("SeriesInstanceUID", d), "errors": errors})
        return None
    if len(got) < len(planes):
        DECODE_FAILED.append({"series": rec.get("SeriesInstanceUID", d), "errors": errors})
        for k, p in enumerate(planes):
            if p is None:
                planes[k] = planes[min(got, key=lambda j: abs(j - k))]
    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)
    if px and np.isfinite(px) and px > 0:
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = h // 2, w // 2
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]
    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)
    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode="bilinear", align_corners=False)
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)
def _atomic_save_npy(path, arr):
    path = Path(path)
    tmp = path.with_name(path.stem + ".tmp.npy")
    np.save(tmp, arr)
    os.replace(tmp, path)
def _atomic_save_json(path, obj):
    path = Path(path)
    tmp = path.with_name(path.stem + ".tmp.json")
    tmp.write_text(json.dumps(obj, indent=2))
    os.replace(tmp, path)
def warn_if_interactive():
    run_type = os.environ.get("KAGGLE_KERNEL_RUN_TYPE")
    log("=" * 70)
    if run_type and run_type.lower() != "batch":
        log(f"RUN TYPE: {run_type!r} - this looks like an interactive Draft Session.")
        log("Files written here are NOT guaranteed to persist once this session ends.")
        log("To keep this output for real, use: Save Version -> Save & Run All (Commit),")
        log("then check the notebook's non-edit page for that version's Output panel.")
    elif run_type is None:
        log("RUN TYPE: unknown (KAGGLE_KERNEL_RUN_TYPE not set) - cannot confirm this "
            "is a committed run.")
    else:
        log(f"RUN TYPE: {run_type!r} - this is a committed run; output should persist "
            f"to this version's Output panel once it finishes.")
    log("=" * 70)
def verify_output(out_dir, split, cache_shape):
    cache_path = Path(out_dir) / f"{split}_cache.dat"
    mask_path = Path(out_dir) / f"{split}_mask.npy"
    order_path = Path(out_dir) / f"{split}_study_order.csv"
    expected_bytes = int(np.prod(cache_shape))
    problems = []
    if not cache_path.is_file():
        problems.append(f"{cache_path.name} missing")
    elif cache_path.stat().st_size != expected_bytes:
        problems.append(f"{cache_path.name} is {cache_path.stat().st_size} bytes, "
                         f"expected {expected_bytes}")
    if not mask_path.is_file():
        problems.append(f"{mask_path.name} missing")
    if not order_path.is_file():
        problems.append(f"{order_path.name} missing")
    return problems
def open_or_create_memmap(path, shape, dtype=np.uint8):
    path = Path(path)
    nbytes = int(np.prod(shape)) * np.dtype(dtype).itemsize
    if path.is_file() and path.stat().st_size == nbytes:
        log(f"resuming from existing {path.name} ({nbytes / 1024 ** 3:.2f} GB)")
        return np.memmap(path, dtype=dtype, mode="r+", shape=shape), True
    path.parent.mkdir(parents=True, exist_ok=True)
    log(f"creating new {path.name} ({nbytes / 1024 ** 3:.2f} GB)")
    return np.memmap(path, dtype=dtype, mode="w+", shape=shape), False
def build_cache(slot_map, cache_slices, img_size, tag, out_dir,
                order_budget_s=ORDER_BUDGET_S, order_cache_path=None,
                flush_every=FLUSH_EVERY):
    studies = sorted(slot_map)
    sidx = {s: i for i, s in enumerate(studies)}
    shape = (len(studies), N_SLOT, cache_slices, img_size, img_size)
    cache_path = Path(out_dir) / f"{tag}_cache.dat"
    mask_path = Path(out_dir) / f"{tag}_mask.npy"
    cache, resumed = open_or_create_memmap(cache_path, shape)
    if resumed and mask_path.is_file():
        prev_mask = np.load(mask_path)
        if prev_mask.shape == (len(studies), N_SLOT):
            mask = prev_mask.astype(np.float32)
        else:
            log(f"{tag}: mask shape mismatch on resume ({prev_mask.shape} vs "
                f"{(len(studies), N_SLOT)}); restarting mask and cache together")
            cache, resumed = open_or_create_memmap(cache_path, shape)
            cache[:] = 0
            mask = np.zeros((len(studies), N_SLOT), np.float32)
            resumed = False
    else:
        mask = np.zeros((len(studies), N_SLOT), np.float32)
    log(f"{tag}: cache {cache.shape} = {cache.nbytes / 1024 ** 3:.1f} GB"
        + (" (resumed)" if resumed else ""))
    all_jobs = [(st, k, plane, slot_map[st][name])
                for st in studies
                for k, (name, plane, _, _) in enumerate(SLOTS)
                if name in slot_map[st]]
    n_job_total = len(all_jobs)
    jobs = [j for j in all_jobs if mask[sidx[j[0]], j[1]] < 0.5]
    if resumed:
        log(f"{tag}: {n_job_total - len(jobs)}/{n_job_total} slots already cached, "
            f"{len(jobs)} remaining")
    seen = {}
    if order_cache_path and Path(order_cache_path).is_file():
        try:
            seen = json.loads(Path(order_cache_path).read_text())
        except (OSError, ValueError):
            seen = {}
    t_ord = time.time()
    ok = 0
    hit = 0
    for _, _, _, rec in jobs:
        e = seen.get(rec["SeriesInstanceUID"])
        if e and len(e["files"]) == len(rec["files"]):
            rec["ordered"] = e["files"]
            ok += int(e["good"])
            hit += 1
    to_order = [j for j in jobs if "ordered" not in j[3]]
    log(f"{tag}: {hit} slot-series ordered from cache, {len(to_order)} to read "
        f"({sum(len(j[3]['files']) for j in to_order)} slice headers)")
    done = 0
    last_log = time.time()
    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:
        futures = {pool.submit(order_slices, rec): rec for _, _, _, rec in to_order}
        for fut in as_completed(futures):
            rec = futures[fut]
            files, good = fut.result()
            rec["ordered"] = files
            ok += int(good)
            done += 1
            if order_cache_path:
                seen[rec["SeriesInstanceUID"]] = {"files": files, "good": bool(good)}
            now = time.time()
            if now - last_log > 15:
                rate = done / max(now - t_ord, 1e-6)
                remain = len(to_order) - done
                eta_s = remain / max(rate, 1e-6)
                log(f"  {tag} ordering {done}/{len(to_order)} "
                    f"({rate:.0f}/s, ~{eta_s / 60:.0f}min remaining)")
                last_log = now
            if now - t_ord > order_budget_s:
                log(f"{tag}: ordering budget spent at {done}/{len(to_order)}; "
                    f"rest keep file order")
                for f in futures:
                    f.cancel()
                break
    if order_cache_path and done:
        tmp = Path(order_cache_path).with_suffix(".tmp")
        tmp.write_text(json.dumps(seen))
        tmp.replace(order_cache_path)
    log(f"{tag}: ordered {ok}/{len(jobs)} by geometry ({len(jobs) - ok} kept arbitrary) "
        f"in {time.time() - t_ord:.0f}s")
    log(f"{tag}: decoding {len(jobs)} slot-series")
    n_failed_before = len(DECODE_FAILED)
    done = 0
    since_flush = 0
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        futures = {pool.submit(read_slot, rec, cache_slices, img_size): (st, k, plane)
                   for st, k, plane, rec in jobs}
        for fut in as_completed(futures):
            st, k, plane = futures[fut]
            done += 1
            since_flush += 1
            try:
                img = fut.result()
            except Exception as exc:
                DECODE_FAILED.append({"series": st, "errors": [f"{type(exc).__name__}: {str(exc)[:120]}"]})
                img = None
            if img is not None:
                cache[sidx[st], k] = normalise_laterality(img, plane, _CURRENT_LAT.get(st)).numpy()
                mask[sidx[st], k] = 1.0
            if since_flush >= flush_every:
                cache.flush()
                _atomic_save_npy(mask_path, mask)
                since_flush = 0
            if done % 4096 < 1:
                log(f"  {tag} {done}/{len(jobs)}")
    cache.flush()
    _atomic_save_npy(mask_path, mask)
    n_failed = len(DECODE_FAILED) - n_failed_before
    log(f"{tag}: {int(mask.sum())}/{n_job_total} slots filled"
        + (f"; {n_failed} series had a slice that would not decode" if n_failed else ""))
    gc.collect()
    return studies, cache, mask
def coverage_report(tag, studies, cache, mask, slot_map, lat_map):
    n_study = len(studies)
    log(f"=== {tag} coverage report ===")
    log(f"studies: {n_study}")
    log(f"overall slot fill rate: {mask.mean():.1%}")
    for k, (name, plane, fluid, fs) in enumerate(SLOTS):
        log(f"  {name:16s} {int(mask[:, k].sum())}/{n_study} ({mask[:, k].mean():.1%})")
    n_lat_resolved = sum(1 for st in studies if lat_map.get(st) is not None)
    log(f"laterality resolved: {n_lat_resolved}/{n_study} "
        f"({n_lat_resolved / max(n_study, 1):.1%})")
    n_chosen = n_unscaled = 0
    for st in studies:
        for name, row in slot_map.get(st, {}).items():
            n_chosen += 1
            px = row.get("px")
            if px is None or not np.isfinite(px) or px <= 0:
                n_unscaled += 1
    if n_chosen:
        log(f"chosen slot-series without usable pixel spacing (not physically cropped): "
            f"{n_unscaled}/{n_chosen} ({n_unscaled / n_chosen:.1%})")
    if DECODE_FAILED:
        reasons = Counter()
        for entry in DECODE_FAILED:
            for e in entry["errors"]:
                reasons[e.split(":")[0]] += 1
        log(f"decode failures: {len(DECODE_FAILED)} series affected")
        for reason, n in reasons.most_common(10):
            log(f"  {reason}: {n}")
    else:
        log("decode failures: none")
_CURRENT_LAT = {}
def process_split(root, split, out_dir, n_slice_target, order_cache_path):
    log(f"walking {split} series directories + reading headers")
    h = walk(root, split)
    if h.empty:
        log(f"{split}: no series found, skipping")
        return None
    h = annotate(h)
    plane_map = h.set_index("SeriesInstanceUID")["Anatomical_Plane"].to_dict()\
        if "Anatomical_Plane" in h.columns else {}
    if not plane_map:
        series_csv = root / f"{split}_series.csv"
        if series_csv.is_file():
            series_meta = pd.read_csv(series_csv)
            plane_map = series_meta.set_index("SeriesInstanceUID")["Anatomical_Plane"].to_dict()
    lat_map = lat_of(h, tag=f"{split} ")
    global _CURRENT_LAT
    _CURRENT_LAT = lat_map
    slot_map = pick_slots(h, plane_map)
    studies, cache, mask = build_cache(
        slot_map, cache_slices=n_slice_target, img_size=CACHE_IMG, tag=split,
        out_dir=out_dir, order_cache_path=order_cache_path,
    )
    pd.Series(studies, name="StudyInstanceUID").to_csv(
        Path(out_dir) / f"{split}_study_order.csv", index=False)
    coverage_report(split, studies, cache, mask, slot_map, lat_map)
    return studies, cache, mask
if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument(
        "--splits", default="train,test",
        help="comma-separated splits to process. Pass '--splits test' to rebuild "
             "ONLY the test cache from whatever test_series/ is present right now - "
             "use this inside the final submission notebook so a swapped-in hidden "
             "test set at grading time actually gets processed, instead of reusing a "
             "frozen cache built earlier against the 3-study example test.csv.")
    ap.add_argument(
        "--out-dir", default="preprocessed",
        help="directory to write {split}_cache.dat / _mask.npy / _study_order.csv / "
             "_MANIFEST.json into")
    ap.add_argument(
        "--cache-slices", type=int, default=None,
        help="force the number of slices cached per slot. MUST match whatever value "
             "was used to build the *other* split's cache (check that run's log for "
             "'-> N slices/slot') - the model's input shape at inference has to match "
             "what it was trained on. If omitted, sized automatically from available "
             "memory, which is only safe when building train and test together in the "
             "same run.")
    args = ap.parse_args()
    splits = [s.strip() for s in args.splits.split(",") if s.strip()]

    OUT_DIR = Path(args.out_dir)
    ORDER_CACHE_PATH = OUT_DIR / "order_cache.json"

    ROOT = find_root()
    log(f"input root: {ROOT}")
    warn_if_interactive()
    OUT_DIR.mkdir(exist_ok=True, parents=True)
    train_df = pd.read_csv(ROOT / "train.csv")
    n_slice_target = args.cache_slices if args.cache_slices is not None else plan_cache(len(train_df))
    manifest_entries = {}
    problems_all = []
    for split in splits:
        result = process_split(ROOT, split, OUT_DIR, n_slice_target, ORDER_CACHE_PATH)
        if result is None:
            continue
        studies, cache, mask = result
        problems = verify_output(OUT_DIR, split, cache.shape)
        problems_all += problems
        cache_path = OUT_DIR / f"{split}_cache.dat"
        mask_path = OUT_DIR / f"{split}_mask.npy"
        manifest_entries[split] = {
            "n_studies": len(studies),
            "cache_shape": list(cache.shape),
            "cache_bytes": cache_path.stat().st_size if cache_path.is_file() else 0,
            "mask_bytes": mask_path.stat().st_size if mask_path.is_file() else 0,
            "verified": len(problems) == 0,
        }
        del cache, mask, studies, result
        gc.collect()
    _atomic_save_json(OUT_DIR / "_MANIFEST.json", {
        "written_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "kaggle_run_type": os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "unknown"),
        "splits": manifest_entries,
    })
    log("=" * 70)
    if problems_all:
        log("SAVE VERIFICATION FAILED:")
        for p in problems_all:
            log(f"  - {p}")
        log("=" * 70)
        raise RuntimeError(
            f"{len(problems_all)} output file(s) missing or the wrong size after the "
            f"run finished - see the list above. Do not trust this run's output.")
    log("SAVE VERIFICATION PASSED - every expected file is on disk at the right size:")
    for split, info in manifest_entries.items():
        log(f"  {split}: {info['n_studies']} studies, "
            f"cache {info['cache_bytes'] / 1024 ** 3:.2f} GB, verified OK")
    warn_if_interactive()
    log("=" * 70)

Writing image_preprocessing.py


In [11]:
!python image_preprocessing.py --splits test --out-dir preprocessed_test_live --cache-slices 3

[    0.0s] input root: /kaggle/input/competitions/rsna-knee-abnormality-detection
[    0.0s] ======================================================================
[    0.0s] RUN TYPE: 'Batch' - this is a committed run; output should persist to this version's Output panel once it finishes.
[    0.0s] ======================================================================
[    0.1s] walking test series directories + reading headers
[    0.2s] test laterality: 1 from the tag, 2 from geometry, 0 unresolved; tag and geometry disagree on 0 (0.0% of the tagged)
[    0.2s] creating new test_cache.dat (0.01 GB)
[    0.2s] test: cache (3, 6, 3, 336, 336) = 0.0 GB
[    0.2s] test: 0 slot-series ordered from cache, 12 to read (469 slice headers)
[    2.0s] test: ordered 12/12 by geometry (0 kept arbitrary) in 2s
[    2.0s] test: decoding 12 slot-series
[    2.3s] test: 12/12 slots filled
[    2.4s] === test coverage report ===
[    2.4s] studies: 3
[    2.4s] overall slot fill rate: 66.7%
[    2.4

In [12]:
%%writefile infer.py
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

sys.path.insert(0, str(Path(__file__).parent))
from dataset import KneeMRICache, TARGETS
from model import KneeMRIModel


def load_model(checkpoint, backbone, dinov2_variant, img_size, device):
    backbone_kwargs = {"variant": dinov2_variant} if backbone == "dinov2" else {}
    model = KneeMRIModel(
        n_labels=len(TARGETS),
        backbone=backbone,
        pretrained=False,
        train_img_size=img_size,
        backbone_kwargs=backbone_kwargs,
    ).to(device)
    ckpt = torch.load(checkpoint, map_location=device)
    model.load_state_dict(ckpt["model"])
    model.eval()
    print(
        f"[INFO] loaded {checkpoint} (epoch={ckpt.get('epoch')}, "
        f"best_auc={ckpt.get('best_auc')})",
        flush=True,
    )
    return model


@torch.no_grad()
def run_inference(model, cache, available, batch_size, device):
    preds = {}
    for start in range(0, len(available), batch_size):
        batch_uids = available[start : start + batch_size]
        idxs = [cache.study_to_idx[u] for u in batch_uids]
        slots = torch.from_numpy(np.asarray(cache.cache[idxs])).to(device)
        mask = torch.from_numpy(cache.mask[idxs].copy()).to(device)
        logits = model(slots, mask)
        probs = torch.sigmoid(logits).cpu().numpy()
        for u, p in zip(batch_uids, probs):
            preds[u] = p
        if start % (batch_size * 10) == 0:
            print(f"[INFO] inferred {start + len(batch_uids)}/{len(available)}", flush=True)
    return preds


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--preprocessed-dir", default="preprocessed")
    ap.add_argument(
        "--checkpoint",
        nargs="+",
        default=["training_output/best.pt"],
        help="one or more best.pt paths - pass multiple (one per fold) to "
        "average their predictions (fold ensembling)",
    )
    ap.add_argument(
        "--test-studies-csv",
        default="test.csv",
        help="the competition's test.csv (single StudyInstanceUID column) "
        "or sample_submission.csv - only the StudyInstanceUID column is used",
    )
    ap.add_argument("--out", default="submission.csv")
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--img-size", type=int, default=224)
    ap.add_argument("--backbone", choices=["resnet50", "dinov2"], default="resnet50")
    ap.add_argument(
        "--dinov2-variant", choices=["small", "base", "large"], default="small"
    )
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[INFO] device={device}", flush=True)

    all_studies = pd.read_csv(args.test_studies_csv)["StudyInstanceUID"].tolist()
    print(f"[INFO] {len(all_studies)} studies required in submission", flush=True)

    cache = KneeMRICache(args.preprocessed_dir, "test")
    available = [s for s in all_studies if s in cache.study_to_idx]
    missing = [s for s in all_studies if s not in cache.study_to_idx]
    if missing:
        print(
            f"[WARNING] {len(missing)} studies have no preprocessed test cache entry "
            f"(likely DICOM decode failures) - these rows will be filled with 0.5 "
            f"per label rather than a real prediction: {missing[:5]}"
            f"{'...' if len(missing) > 5 else ''}",
            flush=True,
        )

    for c in args.checkpoint:
        if not Path(c).is_file():
            raise FileNotFoundError(
                f"{c} not found - train.py must complete and save best.pt for every "
                f"fold you're passing in before inference can run."
            )

    print(f"[INFO] ensembling {len(args.checkpoint)} checkpoint(s)", flush=True)

    sum_preds = {u: np.zeros(len(TARGETS), dtype=np.float64) for u in available}
    for ckpt_path in args.checkpoint:
        model = load_model(
            ckpt_path, args.backbone, args.dinov2_variant, args.img_size, device
        )
        preds = run_inference(model, cache, available, args.batch_size, device)
        for u, p in preds.items():
            sum_preds[u] += p
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

    n_ckpts = len(args.checkpoint)
    preds = {u: p / n_ckpts for u, p in sum_preds.items()}

    rows = []
    for uid in all_studies:
        if uid in preds:
            row = {"StudyInstanceUID": uid}
            row.update({t: float(preds[uid][i]) for i, t in enumerate(TARGETS)})
        else:
            row = {"StudyInstanceUID": uid}
            row.update({t: 0.5 for t in TARGETS})
        rows.append(row)

    sub = pd.DataFrame(rows, columns=["StudyInstanceUID"] + TARGETS)
    if len(sub) != len(all_studies):
        raise RuntimeError(
            f"submission has {len(sub)} rows but {len(all_studies)} "
            f"studies were required - refusing to write a malformed file"
        )
    if sub.isnull().any().any():
        raise RuntimeError("submission contains null values - refusing to write")

    out_path = Path(args.out)
    tmp_path = Path(str(out_path) + ".tmp")
    sub.to_csv(tmp_path, index=False)
    tmp_path.replace(out_path)
    print(
        f"[DONE] wrote {out_path} with {len(sub)} rows ({len(missing)} filled with 0.5, "
        f"averaged over {n_ckpts} checkpoint(s))",
        flush=True,
    )


if __name__ == "__main__":
    main()

Writing infer.py


In [13]:
!python infer.py \
  --preprocessed-dir preprocessed_test_live \
  --checkpoint training_output_dinov2_fold0/best.pt training_output_dinov2_fold1/best.pt training_output_dinov2_fold2/best.pt training_output_dinov2_fold3/best.pt training_output_dinov2_fold4/best.pt \
  --test-studies-csv /kaggle/input/competitions/rsna-knee-abnormality-detection/test.csv \
  --backbone dinov2 \
  --dinov2-variant small \
  --img-size 224 \
  --out submission.csv

[INFO] device=cuda
[INFO] 3 studies required in submission
[INFO] ensembling 5 checkpoint(s)
[INFO] loaded training_output_dinov2_fold0/best.pt (epoch=13, best_auc=0.8143941990556801)
[INFO] inferred 3/3
[INFO] loaded training_output_dinov2_fold1/best.pt (epoch=8, best_auc=0.7886468730338114)
[INFO] inferred 3/3
[INFO] loaded training_output_dinov2_fold2/best.pt (epoch=12, best_auc=0.8275930492690561)
[INFO] inferred 3/3
[INFO] loaded training_output_dinov2_fold3/best.pt (epoch=11, best_auc=0.8087651916191068)
[INFO] inferred 3/3
[INFO] loaded training_output_dinov2_fold4/best.pt (epoch=16, best_auc=0.8040429032045965)
[INFO] inferred 3/3
[DONE] wrote submission.csv with 3 rows (0 filled with 0.5, averaged over 5 checkpoint(s))
